# 1 简介
LangChain 是一套用于构建 AI 智能体（AI Agent）和大语言模型（LLM）应用的开发框架。

核心组件
- LLM：连接 OpenAI、Claude、Gemini 等大模型
- PromptTemplate：管理 Prompt 模板
- Chains：构建多步骤 AI 工作流
- Memory：实现多轮对话记忆
- Tools：调用搜索、数据库、API 等工具
- Agents：让 AI 自动决策与执行任务
- Vector Store：连接向量数据库实现 RAG

| 对比维度 | LangChain | LangGraph |
| :--- | :--- | :--- |
| 定位 | 高层 Agent 框架，开箱即用 | 底层工作流引擎，精细控制 |
| 上手难度 | 低，10 行代码创建 Agent | 中高，需理解图（Graph）概念 |
| 适用场景 | 标准 Agent 应用、快速原型 | 复杂多步骤工作流、多 Agent 协作 |

LangChain 文档： https://python.langchain.com/

# 2 安装
```bash
$ pip install langchain python-dotenv

# LangChain 本身不包含具体的模型实现，你需要根据使用的模型安装对应的提供商包：
pip install langchain-openai # OpenAI
pip install langchain-anthropic # Anthropic
pip install langchain-deepseek # DeepSeek
pip install langchain-google-genai # Google
pip install langchain-ollama # Ollama（本地模型）
```

In [1]:
# 验证 langchain 导入
try:
    import langchain
    print(f"langchain 版本: {langchain.__version__}")
except ImportError:
    print("错误：langchain 未安装，请运行 pip install langchain")

# 验证 langchain-openai 导入
try:
    import langchain_openai
    print("langchain-openai 已安装")
except ImportError:
    print("错误：langchain-openai 未安装，请运行 pip install langchain-openai")

langchain 版本: 1.3.11
langchain-openai 已安装


## 2.1 设置环境变量
使用 .env 文件管理 API Key。在项目根目录创建 .env 文件
```py
# 填入你的 API Key
OPENAI_API_KEY=sk-your-api-key-here

# 如果使用其他模型，也在这里配置
# ANTHROPIC_API_KEY=sk-ant-your-key
# DEEPSEEK_API_KEY=sk-your-key
```
创建 .gitignore 文件，确保 .env 不会被提交：
```py
# .gitignore
.env
__pycache__/
*.pyc
```

In [ ]:
# 在程序开头加载 .env 文件
import os
from dotenv import load_dotenv

# 加载 .env 文件中的环境变量
load_dotenv()

# 验证 API Key 是否加载成功
api_key = os.getenv("DASHSCOPE_API_KEY")
if api_key:
    print(f"API Key 已加载")
else:
    print("警告：未找到API KEY，请检查 .env 文件")

测试一下

In [13]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
import os

llm=ChatOpenAI(
    api_key=os.getenv('DASHSCOPE_API_KEY'),
    model='qwen3.6-flash',
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

# Prompt 模板
prompt = ChatPromptTemplate.from_template(
    "{content}"
)

# 创建 Chain
chain = prompt | llm

# 调用
result = chain.invoke({
    "content": "你好，简单介绍你自己"
})

print(result.content)

你好！我是 Qwen（通义千问），是由阿里巴巴通义实验室研发的大型语言模型。

简单来说，我是一个思维活跃且乐于助人的 AI 伙伴。我可以成为你的得力助手，帮助你处理各种任务，例如：

*   **自由交流与协作**：我支持全球超过 100 种语言，可以随时陪你顺畅地探讨问题。
*   **学习与工作辅助**：无论是撰写创意文案、梳理长文档的核心重点，还是进行复杂的逻辑推理和编写程序代码，我都可以为你提供支持。
*   **多维度分析**：如果你有复杂的图片数据或多步骤任务，我也会尽力帮你清晰地拆解和处理。

无论你需要什么帮助，我都会尽我所能为你提供温暖又清晰的回答。今天有什么我可以帮你的吗？


# 3 Model

## 3.1 init_chat_model() 函数

```py
from langchain.chat_models import init_chat_model

# 完整语法
model = init_chat_model(
    model,                    # str | None：模型名称（provider:model 格式）
    *,
    model_provider=None,      # str | None：单独的模型提供商
    configurable_fields=None, # None | "any" | list[str]：可运行时修改的字段
    config_prefix=None,       # str | None：配置键前缀
    **kwargs,                 # 模型特定参数（temperature、max_tokens 等）
)
```

模型格式
```py
model = init_chat_model("deepseek:deepseek-v4-flash")
model = init_chat_model("anthropic:claude-sonnet-4-5-20250929")
# 如果不指定提供商前缀，LangChain 会尝试从模型名推断：
model = init_chat_model("deepseek-v4-flash") # → openai
model = init_chat_model("claude-sonnet-4-5") # → anthropic
# 等价写法
model = init_chat_model("claude-sonnet-4-5", model_provider="anthropic")
```

kwargs 参数会直接传递给底层模型类，常用的包括：
```py
model = init_chat_model(
    model_provider='openai', # 阿里云兼容openai格式
    api_key=os.getenv('DASHSCOPE_API_KEY'),
    model='qwen3.6-flash',
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1", # 自定义 API 地址
    temperature=0.3, # 控制创造性与确定性，取值范围 0 到 2
    # top_p=0.9,       # 另一种控制随机性的方式。模型只会从累积概率达到 top_p 的词中采样。temperature 或 top_p 选其一，不要同时设置。
    max_tokens=200, # 最大输出token，控制输出长度与成本
    timeout=30, # 单次请求最多等待 30 秒
    max_retries=3, # 失败后最多重试 3 次（总共 4 次请求机会）
    # stop=["\n"],  # 遇到换行就停止，只返回第一个 # stop 参数指定停止序列，模型遇到这些词时会立即停止生成
    seed=42, # 部分模型支持
)
```

设置 configurable_fields 运行时动态指定模型和参数

In [2]:
from langchain.chat_models import init_chat_model
import os

model=init_chat_model(
    model='openai:qwen3.6-flash',
    api_key=os.getenv('DASHSCOPE_API_KEY'),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
    configurable_fields='any', # any 表示所有参数都可配置，或指定列表中的字段可配置["model", "temperature"]
    config_prefix='my', # 配置键前缀
)
result=model.invoke(
    input='hi',
    config={
        'configurable':{
            'my_temperature':0.7, # 动态修改温度
        }
    }
)
print(result.content)

Hi there! How can I help you today? 😊


## 3.2 bind_tools() 函数
让模型知道可用的工具列表。返回包含 tool_calls 的 AIMessage。模型需要支持 function calling。

In [ ]:
# 字典格式的tool描述
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "查询指定城市的天气",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "城市名称，如 杭州、北京"
                    }
                },
                "required": ["city"]
            }
        }
    }
]

model_bind_tools=model.bind_tools(tools)
result=model_bind_tools.invoke("杭州今天天气怎么样？")

print(result.tool_calls)


[{'name': 'get_weather', 'args': {'city': '杭州'}, 'id': 'call_d9083f1e3ba54d938e09ff08', 'type': 'tool_call'}]


In [11]:
from pydantic import BaseModel,Field

# Pydantic格式的tool描述
# 相比于字典形式，它提供了类型安全、自动校验，而且 LangChain 会自动从类名和 Field 描述生成工具描述。
class GetWeather(BaseModel):
    city:str=Field(description="城市名称，如 杭州、北京")

class Calculator(BaseModel):
    """执行数学计算"""
    expression: str = Field(description="要计算的数学表达式，如 '(3 + 5) * 2'")

model_bind_tools=model.bind_tools([GetWeather,Calculator])
result=model_bind_tools.invoke('北京今天多少度？顺便算一下 5 * 45')

for tc in result.tool_calls:
    print(f'工具：{tc["name"]} 参数：{tc["args"]} ID:{tc["id"]}') # 这里只调用了一次模型，因此模型给出工具调用请求

工具：GetWeather 参数：{'city': '北京'} ID:call_820964e62f8f44048115926f
工具：Calculator 参数：{'expression': '5 * 45'} ID:call_5495c59eb3884022bb2d0515


# 4 Message
LangChain 四种核心消息类型

| 类型 | 角色 | 说明 | 典型内容 |
| :--- | :--- | :--- | :--- |
| `HumanMessage` | 用户 | 用户发送的消息 | "今天天气怎么样？" |
| `AIMessage` | AI 助手 | 模型的回复，可能包含 `tool_calls` | "今天杭州晴天，25°C" |
| `SystemMessage` | 系统 | 系统指令，定义 AI 的角色和行为规则 | "你是一个专业的天气助手" |
| `ToolMessage` | 工具 | 工具执行后的返回结果 | "晴，25°C，湿度 60%" |

## 4.1 HumanMessage

In [13]:
from langchain.messages import HumanMessage
from langchain.chat_models import init_chat_model
import os

model=init_chat_model(
    model='openai:qwen3.6-flash',
    api_key=os.getenv('DASHSCOPE_API_KEY'),
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

# msgs=HumanMessage(content='什么是langchain?')
# print(msgs.type) # human
# print(msgs.content) # 什么是langchain?

msgs = [
    HumanMessage(content="你好"),
    HumanMessage(content='简要介绍什么是langchain?'),
    HumanMessage(content="一句话回答langchain与langgragh的核心区别"),
]
result=model.invoke(msgs)
print(result.content)

你好！

**LangChain 简介**  
LangChain 是一个开源的 LLM 应用开发框架，提供了一套标准化的抽象与组件（如提示词管理、链式调用、智能体、记忆模块、工具与检索器接口等），旨在降低连接大语言模型与外部数据/API的门槛，帮助开发者高效构建检索增强生成（RAG）、多步推理、对话系统等复杂 AI 应用。

**LangChain 与 LangGraph 的核心区别（一句话）**  
LangChain 侧重通用的 LLM 应用模块化编排，而 LangGraph 专注基于有向图结构实现带状态的循环工作流与智能体精确控制。


In [ ]:
# 四种等价HumanMessage方式
# 方式 1：标准构造
msg1 = HumanMessage(content="你好")

# 方式 2：元组快捷方式 (role, content)
msg2 = ("user", "你好")
msg3 = ("human", "你好")

# 方式 3：字典快捷方式
msg4 = {"role": "user", "content": "你好"}

## 4.2 AIMessage

In [ ]:
from langchain.messages import AIMessage

msg=AIMessage(content='LangChain 是一个开源框架，专为简化大语言模型（LLM）应用开发而设计。')
# print(msg.content) # LangChain 是一个开源框架，专为简化大语言模型（LLM）应用开发而设计。
# print(msg.type) # ai

msg_tools=AIMessage(
    content="",  # 工具调用时 content 通常为空
    tool_calls=[
        {
            "name": "get_weather",
            "args": {"city": "杭州"},
            "id": "call_abc123",
            "type": "tool_call",
        }
    ]
)
# print(msg_tools.content) # 
# print(msg_tools.tool_calls) # [{'name': 'get_weather', 'args': {'city': '杭州'}, 'id': 'call_abc123', 'type': 'tool_call'}]

In [14]:
# AI回复的内容还包括其他信息，如token，模型，结束原因等

# AIMessage 包含丰富的元数据
print(f"{result.content}\n")
print(f"消息ID: {result.id}")
print(f"模型名: {result.response_metadata.get('model_name')}")
print(f"完成原因: {result.response_metadata.get('finish_reason')}")

# usage_metadata 包含 Token 用量信息
if result.usage_metadata:
    print(f"输入 tokens: {result.usage_metadata.get('input_tokens')}")
    print(f"输出 tokens: {result.usage_metadata.get('output_tokens')}")
    print(f"总计 tokens: {result.usage_metadata.get('total_tokens')}")

你好！

**LangChain 简介**  
LangChain 是一个开源的 LLM 应用开发框架，提供了一套标准化的抽象与组件（如提示词管理、链式调用、智能体、记忆模块、工具与检索器接口等），旨在降低连接大语言模型与外部数据/API的门槛，帮助开发者高效构建检索增强生成（RAG）、多步推理、对话系统等复杂 AI 应用。

**LangChain 与 LangGraph 的核心区别（一句话）**  
LangChain 侧重通用的 LLM 应用模块化编排，而 LangGraph 专注基于有向图结构实现带状态的循环工作流与智能体精确控制。

消息ID: lc_run--019f02f8-a762-78c0-bbd9-7eb31743dcf0-0
模型名: qwen3.6-flash
完成原因: stop
输入 tokens: 29
输出 tokens: 998
总计 tokens: 1027


## 4.3 SystemMessage

In [15]:
from langchain.messages import SystemMessage

msgs=[
    SystemMessage(content='you are a helpful assistant'), # 用于设定 AI 的行为、角色和约束。它放在消息列表的最前面，指导模型如何回复。
    HumanMessage(content='hello'),
]

result = model.invoke(msgs)
print(result.content)

Hello! How can I assist you today?


## 4.4 ToolMessage

In [17]:
from langchain.messages import ToolMessage,AIMessage

msgs=[
    HumanMessage(content='广州天气怎么样'),
    AIMessage(
        content='',
        tool_calls=[
            {
                'name':'get_weather',
                'args':{
                    'city':'广州'
                },
                'id':'call_123',
                'type':'tool_call'
            }
        ]
    ),
    ToolMessage(
        content='晴，25°C，湿度 60%',
        tool_call_id='call_123', # 与 tool_call 的 id 对应
        name='get_weather'
    )
]

result=model.invoke(msgs)
print(result.content)

广州今天天气晴朗，气温约 25°C，相对湿度为 60%，整体气候较为舒适。白天阳光充足，建议做好防晒措施；早晚温差适中，可适当添减衣物。如需了解未来几天的天气变化或具体时段预报，可随时告诉我！ 😊


## 4.5 流式输出

In [ ]:
# stream() 返回的是 AIMessageChunk 迭代器
for chunk in model.stream(input='简单介绍你自己'):
    print(chunk.content,end='',flush=True) 

我是 Qwen（通义千问），由阿里巴巴集团旗下通义实验室自主研发的大语言模型。我专注于为你提供清晰、实用的协助，支持问答解答、文本创作、编程开发、逻辑推理、数据分析及多语言交流等任务。如果你有具体需求或想一起探讨问题，随时告诉我！

## 4.6 ContentBlock
ContentBlock 内容块

| 类型 | 说明 | 用途 |
| :--- | :--- | :--- |
| PlainTextContentBlock | 纯文本内容 | 普通文字消息 |
| ImageContentBlock | 图片内容（base64 或 URL） | 多模态模型的图片输入 |
| ToolCall | 工具调用请求 | AI 请求调用工具 |

In [ ]:
# 当你只需要发送纯文本时，直接传字符串即可，LangChain 会自动处理。只有当你需要在单条消息中混合文本和图片时，才需要手动构建 ContentBlock 列表。
from langchain.messages import PlainTextContentBlock,ImageContentBlock

msg=HumanMessage(content=
        [
            PlainTextContentBlock(text='图片里是什么'),
            ImageContentBlock(url='https://example.com/photo.jpg')
        ]
)

# print(f"消息类型: {type(msg.content)}") # <class 'list'>
# print(f"内容块数量: {len(msg.content)}") # 2

## 4.7 多模态消息

In [ ]:
msg=[
    HumanMessage(
        content=[
            {'type':'text','text':'图片里是什么'},
            {'type':'image_url','image_url':{'url':'https://i-blog.csdnimg.cn/blog_migrate/558b42fa940e2bdc2c5f4060572a11e3.png','detail':'auto'}}  # 可选：low, high, auto
        ]
    )
]
# 或
# msg=[
#     {
#         "role": "user",
#         "content": [
#             {"type": "text", "text": "这张图描述了什么内容？"},
#             {
#                 "type": "image_url",
#                 "image_url": {"url": "https://i-blog.csdnimg.cn/blog_migrate/558b42fa940e2bdc2c5f4060572a11e3.png"}
#             }
#         ]
#     }
# ]

result=model.invoke(msg)
print(result.content)

这张图片展示了**四只不同品种的狗**，它们分别位于画面的四个象限中，背景都是**蓝天、白云和绿色草地**的自然户外场景。

具体来看：

| 位置 | 狗的品种 | 特征 |
|:---|:---|:---|
| **左上** | 看起来像**白色秋田犬**或类似的尖耳犬 | 毛色偏乳白，耳朵竖立，张嘴吐舌，表情开心 |
| **右上** | **柴犬** | 典型的棕白毛色，耳朵竖立，同样张嘴微笑 |
| **左下** | **萨摩耶犬** | 纯白色蓬松长毛，"微笑天使"的典型表情 |
| **右下** | 可能是**白色柴犬**或**北海道犬** | 纯白色，耳朵竖立，吐舌微笑 |

这四张图风格非常统一，应该是用**AI绘画工具**（如Stable Diffusion、Midjourney等）生成的系列作品，展示了不同犬种在相同场景下的效果。画面右下角还有"CSDN @熊猫Jay"的水印，说明这可能是一篇技术博客中展示AI生成图像的示例图。


## 4.8 trim_messages()
裁剪消息历史

In [ ]:
from langchain.messages import trim_messages
messages = [
    SystemMessage(content="你是友好的 AI 助手"),
    HumanMessage(content="Python 怎么入门？"),
    AIMessage(content="Python 入门可以从基础知识开始..."),
    HumanMessage(content="有推荐的 IDE 吗？"),
    AIMessage(content="推荐 VS Code 或 PyCharm..."),
    HumanMessage(content="如何安装第三方库？"),
    AIMessage(content="使用 pip install 命令..."),
    HumanMessage(content="NumPy 是什么？"),
    AIMessage(content="NumPy 是一个科学计算库..."),
    HumanMessage(content="pandas 和 NumPy 有什么区别？"),
]

trimed_messages=trim_messages(
    messages=messages,
    max_tokens=1000,           # 最多保留 1000 tokens
    strategy="last",           # 保留最后的系统消息 + 最近的对话("first"：保留 system 消息 + 最早的对话)
    token_counter=model,       # 使用模型的 token 计数方式
    include_system=True,       # 始终保留 SystemMessage
    start_on="human",          # 裁剪后以 human 消息开头
)

## 4.9 RemoveMessage()
删除特定消息

通常配合 AgentState 的 add_messages reducer 使用。在 middleware 或 after_model 钩子中返回 RemoveMessage 可以动态清理消息历史。

In [ ]:
from langchain.messages import RemoveMessage
messages = [
    HumanMessage(content="你好", id="msg_1"),
    AIMessage(content="你好！有什么可以帮你的？", id="msg_2"),
    HumanMessage(content="帮我查天气", id="msg_3"),
]

removal=RemoveMessage(id='msg_3')

# print(f"{removal.id}") # msg_3

# 5 Agent

## 5.1 create_agent() 函数
```py
from langchain.agents import create_agent

agent = create_agent(
    model,                     # str | BaseChatModel：语言模型
    tools=None,                # Sequence：工具列表
    *,
    system_prompt=None,        # str | SystemMessage：系统提示
    middleware=(),             # Sequence[AgentMiddleware]：中间件列表
    response_format=None,      # ResponseFormat | type：结构化输出配置
    state_schema=None,         # type[AgentState]：自定义状态结构
    context_schema=None,       # type：运行时上下文结构
    checkpointer=None,         # Checkpointer：对话持久化
    store=None,                # BaseStore：跨会话存储
    interrupt_before=None,     # list[str]：在哪些节点前暂停
    interrupt_after=None,      # list[str]：在哪些节点后暂停
    debug=False,               # bool：是否输出详细日志
    name=None,                 # str：Agent 名称
    cache=None,                # BaseCache：缓存配置
)
```

model参数
```py
# 方式 1
# create_agent() 内部会自动处理模型初始化、工具绑定、结构化输出等逻辑。
agent = create_agent(
    model="deepseek:deepseek-v4-flash",
)

# 方式 2
# 适合需要精细控制模型参数的，需要在 Agent 之外也使用同一个模型实例的场景。
model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0.3, max_tokens=500)
agent = create_agent(
    model=model,
)

# 方式 3：传已绑定工具的模型实例
# 不常用，通常让 create_agent 自己管理工具绑定
model_with_tools = init_chat_model("deepseek:deepseek-v4-flash").bind_tools([...])
```

tools参数：
```py
from langchain.tools import tool
from langchain.agents import create_agent

# 格式 1：@tool 装饰的函数（最常用）
@tool
def search_course(keyword: str) -> str:
    """搜索"""
    return f"搜索结果：{keyword} "


# 格式 2：Pydantic BaseModel 类
from pydantic import BaseModel, Field

class WeatherQuery(BaseModel):
    """查询天气"""
    city: str = Field(description="城市名称")


# 格式 3：字典（描述远程工具或内置工具）
mcp_tool = {
    "type": "mcp",
    "server_label": "weather_server",
    "server_url": "https://weather.example.com/sse",
    "allowed_tools": ["get_forecast"],
}

# 混合使用
agent = create_agent(
    model="deepseek:deepseek-v4-flash",
    tools=[search_course, WeatherQuery, mcp_tool],
)
```

create_agent() 返回一个 CompiledStateGraph 对象，这是 LangGraph 的编译后的图，提供了多种运行方式：

| 方法 | 说明 | 适用场景 |
| :--- | :--- | :--- |
| `invoke(input, config)` | 同步运行，等待完整结果 | 脚本、简单接口 |
| `ainvoke(input, config)` | 异步运行，等待完整结果 | Web 服务 |
| `stream(input, config, stream_mode)` | 同步流式运行 | 实时展示中间步骤 |
| `astream(input, config, stream_mode)` | 异步流式运行 | WebSocket、SSE |
| `get_state(config)` | 获取当前状态 | 查看/恢复对话状态 |
| `update_state(config, values)` | 更新状态 | 手动修改对话状态 |

# 6 Tool

## 6.1  @tool 装饰器
在函数前加上 @tool 装饰器，函数就变成了一个工具。

支持多种参数类型，包括 int、float、bool 和枚举值。可以为工具参数设置默认值，可以让工具用起来更灵活

将定义好的工具传给 create_agent() 的 tools 参数，Agent 就能使用它了

一个 Agent 可以注册多个工具，模型会自动判断何时使用哪个工具

注：函数的文档字符串（docstring）会自动成为工具的描述。Agent 依赖这个描述来判断"这个工具能做什么"和"什么情况下应该调用它"。文档字符串写得越清晰，Agent 使用工具就越准确。在描述中说明参数含义、函数功能和使用场景。

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.messages import HumanMessage
import os
# 使用 @tool 装饰器，把普通 Python 函数变成 Agent 可以调用的工具
@tool
def get_weather(city:str)->str:
    """查询指定城市的天气情况。

    Args:
        city: 城市名称，如 "杭州"、"北京"
    """
    # 实际项目中可以替换为真实的天气 API 调用
    weather_data={
        "杭州": "晴，25°C，湿度 60%",
        "北京": "多云，18°C，湿度 45%",
        "上海": "小雨，22°C，湿度 80%",
    }
    return weather_data.get(city,f'未找到{city}的天气信息')

@tool
def calculate(expression: str) -> str:
    """执行数学计算。支持加减乘除等基本运算。

    Args:
        expression: 数学表达式，如 "3 * 7 + 2"
    """
    try:
        # 安全地计算数学表达式
        result = eval(expression, {"__builtins__": {}}, {})
        return f"计算结果: {expression} = {result}"
    except Exception as e:
        return f"计算错误: {e}"
    
model = init_chat_model(
    model_provider='openai',
    api_key=os.getenv('DASHSCOPE_API_KEY'),
    model='qwen3.6-flash',
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)
agent = create_agent(
    model=model,
    tools=[get_weather,calculate],
    system_prompt='你是一个能使用工具的助手'
)

inputs={'messages':[HumanMessage(content='杭州和北京今天温差多少度？')]}
result=agent.invoke(inputs)
# print(result)
# 查看消息历史（包含 AI 的工具调用和工具返回结果）
print("=== 完整消息历史 ===")
for msg in result["messages"]:
    print(f"[{msg.type}] {msg.content}")

print("\n=== 最终回复 ===")
print(result["messages"][-1].content) # 最后一条 AI 消息就是最终答案

=== 完整消息历史 ===
[human] 杭州和北京今天温差多少度？
[ai] 
[tool] 晴，25°C，湿度 60%
[tool] 多云，18°C，湿度 45%
[ai] 
[tool] 计算结果: 25 - 18 = 7
[ai] 杭州今天的天气是晴，气温25°C；北京今天多云，气温18°C。两座城市今天的温差是7度。

=== 最终回复 ===
杭州今天的天气是晴，气温25°C；北京今天多云，气温18°C。两座城市今天的温差是7度。


In [ ]:
# 异步运行 Agent
import asyncio
from langchain.messages import HumanMessage


async def main():
    # ainvoke() 是 invoke() 的异步版本
    inputs = {"messages": [HumanMessage(content="杭州天气怎么样？")]}
    result = await agent.ainvoke(inputs)
    print(result["messages"][-1].content)


# 运行异步函数
asyncio.run(main())

工具定义方式

| 方式 | 代码量 | 适用场景 | 示例 |
| :--- | :--- | :--- | :--- |
| `@tool` 装饰器 | 最少 | 简单到中等复杂度的工具 | 大多数场景 |
| `@tool` + `args_schema` | 中等 | 需要精细参数校验的工具 | API 封装、数据库操作 |
| Pydantic 类作为工具 | 较多 | 复杂业务逻辑的工具 | 内部包含状态的工具 |
| 字典格式 | 最少（不推荐） | 描述远程/内置工具 | MCP 工具、服务端工具 |

## 6.2 args_schema 自定义参数校验

In [7]:
from pydantic import BaseModel,Field

class GetWeather(BaseModel):
    city:str=Field(description="城市名称，如 杭州、北京")

class Calculator(BaseModel):
    """执行数学计算"""
    expression: str = Field(description="要计算的数学表达式，如 '(3 + 5) * 2'")

@tool(args_schema=GetWeather)
def get_weather(city:str)->str:
    """查询指定城市的天气情况。

    Args:
        city: 城市名称，如 "杭州"、"北京"
    """
    # 实际项目中可以替换为真实的天气 API 调用
    weather_data={
        "杭州": "晴，25°C，湿度 60%",
        "北京": "多云，18°C，湿度 45%",
        "上海": "小雨，22°C，湿度 80%",
    }
    return weather_data.get(city,f'未找到{city}的天气信息')

@tool(args_schema=Calculator)
def calculate(expression: str) -> str:
    """执行数学计算。支持加减乘除等基本运算。

    Args:
        expression: 数学表达式，如 "3 * 7 + 2"
    """
    try:
        # 安全地计算数学表达式
        result = eval(expression, {"__builtins__": {}}, {})
        return f"计算结果: {expression} = {result}"
    except Exception as e:
        return f"计算错误: {e}"

model_bind_tools=model.bind_tools([GetWeather,Calculator])
# model_bind_tools=model.bind_tools([get_weather,calculate])
result=model_bind_tools.invoke('北京今天多少度？顺便算一下 5 * 45')

if result.tool_calls:
    for tc in result.tool_calls:
        print(f'工具：{tc["name"]}  参数{tc["args"]}  ID:{tc["id"]}')

agent=create_agent(
    model=model,
    tools=[get_weather,calculate],
    system_prompt='你是一个能使用工具的助手'
)

inputs={'messages':[HumanMessage(content='杭州和北京今天温差多少度？')]}
result=agent.invoke(inputs)

print("=== 完整消息历史 ===")
for msg in result["messages"]:
    print(f"[{msg.type}] {msg.content}")

print("\n=== 最终回复 ===")
print(result["messages"][-1].content)

工具：GetWeather  参数{'city': '北京'}  ID:call_053b37bf5eae485c83762b02
工具：Calculator  参数{'expression': '5 * 45'}  ID:call_1b91f873cb294bb3afe7c4cb
=== 完整消息历史 ===
[human] 杭州和北京今天温差多少度？
[ai] 
[tool] 晴，25°C，湿度 60%
[tool] 多云，18°C，湿度 45%
[ai] 
[tool] 计算结果: 25 - 18 = 7
[ai] 杭州今天的温度是25°C，北京的温度是18°C。两地的温差为7度。

=== 最终回复 ===
杭州今天的温度是25°C，北京的温度是18°C。两地的温差为7度。


# 7 Tool 高级

## 7.1 return_direct
默认情况下，工具执行后结果会返回给模型，模型再基于工具结果生成最终回复。但有时工具结果本身就是你想要的最终答案。

设置 return_direct=True 后，工具执行完就立即结束 Agent 循环，工具返回内容直接作为最终输出。

In [ ]:
@tool(return_direct=True)
def calculate(expression: str) -> str:
    """执行数学计算。支持加减乘除等基本运算。

    Args:
        expression: 数学表达式，如 "3 * 7 + 2"
    """
    try:
        # 安全地计算数学表达式
        result = eval(expression, {"__builtins__": {}}, {})
        return f"计算结果: {expression} = {result}"
    except Exception as e:
        return f"计算错误: {e}"

model = init_chat_model(
    model_provider='openai',
    api_key=os.getenv('DASHSCOPE_API_KEY'),
    model='qwen3.6-flash',
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

agent_direct=create_agent(
    model=model,
    tools=[calculate],
    system_prompt='你是一个能使用工具的助手'
)

result = agent_direct.invoke({'messages':[HumanMessage(content='计算3乘7加2')]})
print(result["messages"][-1].content)

计算结果: 3 * 7 + 2 = 23


## 7.2 ToolException
使用 ToolException 抛出明确的工具异常，让 Agent 知道出了问题。

In [ ]:
from langchain.tools import tool,ToolException

@tool
def get_user_info(id:int)->str:
    """根据用户 ID 查询用户信息。

    Args:
        user_id: 用户 ID，必须是正整数
    """
    users = {
        1: "张三",
        2: "李四",
    }
    if id<=0:
        raise ToolException(f'用户 ID 必须为正整数，收到了: {id}')
    if id not in users:
        raise ToolException(f'未找到 ID 为 {id} 的用户')
    return users.get(id)

try:
    print(get_user_info.invoke({'id':-1}))
except ToolException as e:
    print(e)

try:
    print(get_user_info.invoke({'id':3}))
except ToolException as e:
    print(e)

print(get_user_info.invoke({'id':2}))


用户 ID 必须为正整数，收到了: -1
未找到 ID 为 3 的用户
李四


## InjectedState
默认情况下，工具只能通过参数接收模型传来的数据。但有时工具需要知道当前对话的上下文——比如之前的对话历史、用户已确认的信息等。

InjectedState 让工具可以直接读取 Agent 的完整状态。

## InjectedStore

## 待续

# System Prompt

## 系统提示词设计

| 要素 | 说明 |
| :--- | :--- |
| **角色定义** | 明确 AI 的身份和职责 |
| **行为准则** | 约束回复的风格和边界 |
| **工具使用指引** | 告诉模型何时使用哪些工具 | 
| **边界约束** | 明确什么不能做 |
| **格式要求** | 指定回复的格式（可选） |

## @dynamic_prompt
静态 system_prompt 对所有用户一视同仁。但实际应用中，你可能需要根据用户信息、对话上下文、时间等动态调整提示词。

@dynamic_prompt 装饰器让你在每次模型调用前动态生成 system_prompt。

@dynamic_prompt 在每次模型调用前都会执行，所以提示词可以随对话推进而变化。但注意不要在里面做太重的计算，否则会影响响应速度。

如果同时设置了 create_agent() 的 system_prompt 和 @dynamic_prompt middleware，middleware 的优先级更高

In [29]:
from langchain.agents.middleware import dynamic_prompt
from langchain.agents.middleware.types import ModelRequest
from langchain.tools import tool
from langchain.agents import create_agent

@tool
def search(keyword:str)->str:
    """通过关键词搜索信息"""
    return f'关于{keyword}的信息'

@dynamic_prompt
def personal_prompt(request:ModelRequest)->str:
    messages=request.state.get('messages',[])
    base_prompt='你是一个个人助手'
    if len(messages)<=2:
        base_prompt+='用户刚开始对话，请先热情问候'
    elif len(messages)>10:
        base_prompt+='对话已经比较长了，回答要尽量简洁'
    else:
        base_prompt+='根据用户的问题判断是否调用工具来回答'
    return base_prompt

model = init_chat_model(
    model_provider='openai',
    api_key=os.getenv('DASHSCOPE_API_KEY'),
    model='qwen3.6-flash',
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)

dynamic_prompt_agent=create_agent(
    model=model,
    tools=[search],
    middleware=[personal_prompt]
)

result=dynamic_prompt_agent.invoke({'messages':[HumanMessage(content='你好')]})
print(result['messages'][-1].content) # 第一轮回复

你好呀！👋 很高兴见到你！

我是你的专属人工智能助手。非常高兴能和你开始这次对话！😄 无论你是想聊聊心事、询问知识、寻找灵感，还是仅仅想发发呆，我都随时在这里陪着你。

今天过得怎么样呢？有没有什么我可以帮到你的？✨


结合运行时上下文

In [ ]:
from datetime import datetime
# @dynamic_prompt 的 request 参数提供了丰富的信息
@dynamic_prompt
def context_aware_prompt(request:ModelRequest)->str:
    context = request.runtime.context # 从 runtime.context 获取用户信息
    user_name = context.get("user_name", "同学") if context else "同学"
    user_level = context.get("user_level", "入门") if context else "入门"

    # 获取当前时间
    now = datetime.now()

    # 获取当前消息数
    messages = request.state.get("messages", [])

    prompt = f"""你是一位学习顾问。

当前时间：{now.strftime('%Y年%m月%d日 %H:%M')}
用户信息：{user_name}，{user_level} 级别

## 行为准则
- 称呼用户为"{user_name}"
- 根据用户级别（{user_level}）推荐合适难度的课程
- 回答要友好但不啰嗦"""

    # 长对话时追加简化提示
    if len(messages) > 20:
        prompt += "\n- 对话很长了，回答尽量精简"

    return prompt

# 流式输出

In [ ]:
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
import os

model = init_chat_model(
    model_provider='openai',
    api_key=os.getenv('DASHSCOPE_API_KEY'),
    model='qwen3.6-flash',
    base_url="https://dashscope.aliyuncs.com/compatible-mode/v1",
)
agent=create_agent(
    model=model,
    system_prompt='你是一个友好的助手'
)
for chunk,metadata in agent.stream( # metadata 包含了这个 chunk 的来源信息
    {'messages':[HumanMessage(content='介绍一下你自己')]},
    stream_mode='messages' # stream_mode="messages" 逐个 Token 返回
):
    print(chunk.content,end='',flush=True)

你好！我是通义千问（Qwen），由阿里巴巴集团旗下通义实验室自主研发的大语言模型。我旨在成为你可靠、高效且贴心的智能助手。

我可以帮你做很多事情，比如：
💡 **解答疑问**：覆盖科学、技术、文化、生活等多个领域的知识问答
✍️ **内容创作**：撰写文章、邮件、报告、故事，或进行润色与翻译
💻 **编程协助**：写代码、debug、解释逻辑，支持多种编程语言
📊 **分析与规划**：梳理信息、总结重点、制定计划或提供学习建议
🗣️ **日常交流**：随时陪你聊天、 brainstorm 想法，或者只是简单聊聊

无论是工作学习中的具体问题，还是生活里的碎碎念，都可以随时告诉我。你会希望我先从哪件小事开始帮你呢？😊

In [ ]:
from langchain.tools import tool
@tool
def get_weather(city:str)->str:
    """查询指定城市的天气情况。

    Args:
        city: 城市名称，如 "杭州"、"北京"
    """
    weather_data={
        "杭州": "晴，25°C，湿度 60%",
        "北京": "多云，18°C，湿度 45%",
        "上海": "小雨，22°C，湿度 80%",
    }
    return weather_data.get(city,f'未找到{city}的天气信息')

agent=create_agent(
    model=model,
    tools=[get_weather],
)

for chunk in agent.stream(
    {'messages':[HumanMessage("上海天气怎么样")]},
    stream_mode='updates' # 逐步查看 Agent 执行过程
): 
    for node_name, update in chunk.items():
        print(f"[{node_name}]", end=" ")
        if "messages" in update:
            for msg in update["messages"]:
                if msg.type == "ai":
                    if hasattr(msg, 'tool_calls') and msg.tool_calls:
                        calls = [tc['name'] for tc in msg.tool_calls]
                        print(f"请求调用: {calls}")
                    elif msg.content:
                        print(f"{msg.content}")
                elif msg.type == "tool":
                    print(f"[{msg.name}]: {msg.content}")

[model] 请求调用: ['get_weather']
[tools] [get_weather]: 小雨，22°C，湿度 80%
[model] 上海当前天气为小雨，气温22°C，湿度80%。


stream_mode 可以组合使用，如 stream_mode=["updates", "custom", "messages"]。但过多的模式会增加流中的事件量，建议按需选择。

| 模式 | 粒度 | 迭代对象 | 典型用途 |
| :--- | :--- | :--- | :--- |
| **messages** | Token 级 | `(AIMessageChunk, metadata)` | 打字效果、实时聊天 |
| **updates** | 节点级 | `{node_name: state_update}` | 展示思考过程 |
| **values** | 节点级（全量） | 完整 `state` | 状态快照、调试 |
| **custom** | 自定义 | 任意 `dict` | 进度通知、状态推送 |
| **debug** | 详细 | 调试信息 | 开发阶段排查问题 |

## 异步流式输出

In [ ]:
import asyncio

async def main():
    agent=create_agent(model=model)
    responsse=''
    async for chunk,metadata in agent.astream(
        {'messages':[HumanMessage('你好')]},
        stream_mode='messages'
    ):
        if chunk.content:
            response+=chunk.content
            print(chunk.content,end='',flush=True)

asyncio.run(main())

## FastAPI 集成示例

In [ ]:
from fastapi import FastAPI
from fastapi.responses import StreamingResponse

app = FastAPI()

@app.get("/chat")
async def chat(message: str):
    """聊天接口，返回 SSE 流式响应"""
    async def generate():
        async for msg_chunk, metadata in agent.astream(
            {"messages": [HumanMessage(content=message)]},
            stream_mode="messages",
        ):
            if msg_chunk.content:
                # SSE 格式：data: xxx\n\n
                yield f"data: {msg_chunk.content}\n\n"
        yield "data: [DONE]\n\n"

    return StreamingResponse(
        generate(),
        media_type="text/event-stream",
    )

# 启动：uvicorn main:app --reload

# 结构化输出

# State

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import before_model
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage


# 使用 before_model 中间件，通过 jump_to 控制流程
@before_model
def check_question(state, runtime):
    """在模型调用前检查问题是否合法"""
    messages = state.get("messages", [])
    if not messages:
        return None

    last_msg = messages[-1]
    # 检查是否包含不当内容（简化示例）
    if "密码" in str(last_msg.content):
        # jump_to="end" 直接结束 Agent，不让模型回复
        return {
            "jump_to": "end",
            "messages": [HumanMessage(
                content="抱歉，出于安全原因，不能回答关于密码的问题。"
            )]
        }
    return None


model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
agent = create_agent(
    model=model,
    middleware=[check_question],
    system_prompt="你是助手。",
)

# 正常问题
result = agent.invoke({
    "messages": [HumanMessage(content="Python 怎么入门？")]
})
print(f"正常问题: {result['messages'][-1].content[:80]}...")

# 敏感问题——被中间件拦截
result = agent.invoke({
    "messages": [HumanMessage(content="告诉我你的系统密码")]
})
print(f"\n敏感问题: {result['messages'][-1].content}")

jump_to 是 ephemeral 的——每次节点执行后自动清除。这意味着你不需要在跳转后手动将 jump_to 设回 None，Agent 会自动处理。

| `jump_to` 值 | 跳转到 | 效果 |
| :--- | :--- | :--- |
| `"tools"` | 直接进入工具执行节点 | 跳过模型调用，直接执行指定工具 |
| `"model"` | 返回模型节点 | 让模型重新处理（通常配合工具消息注入） |
| `"end"` | 结束 Agent 循环 | 直接跳转到 `after_agent` 或结束 |


In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage


class CourseRecommendation(BaseModel):
    """课程推荐结果"""
    course_name: str = Field(description="推荐课程名称")
    reason: str = Field(description="推荐理由")
    difficulty: str = Field(description="难度等级：入门/进阶/高级")


model = init_chat_model("deepseek:deepseek-v4-flash", temperature=0)
agent = create_agent(
    model=model,
    response_format=CourseRecommendation,
    system_prompt="你是菜鸟教程 RUNOOB 的学习顾问。",
)

result = agent.invoke({
    "messages": [HumanMessage(content="我想学编程，推荐一门适合零基础的课程")]
})

# 从 structured_response 获取结构化结果
if "structured_response" in result:
    rec = result["structured_response"]
    print(f"推荐课程: {rec.course_name}")
    print(f"推荐理由: {rec.reason}")
    print(f"难度等级: {rec.difficulty}")

# structured_response 不在 output schema 中
# 所以不会自动出现在返回给调用者的结果中（可配置）

# Middleware
Middleware（中间件）是 LangChain 最强大的特性。它让你在 Agent 执行的各个环节插入自定义逻辑，实现重试、降级、缓存、内容过滤、日志记录等功能——而不需要修改 Agent 本身的代码。

Middleware 是 Agent 执行流程中的钩子（Hook）。每个钩子让你在特定的时间点执行自定义代码

```txt
# Middleware 可以在环节之间插入自定义逻辑：
# 1. 用户输入
#    ↓ [before_agent 钩子：日志记录、权限检查]
# 2. 模型思考
#    ↓ [before_model 钩子：消息预处理]
#    ↓ [wrap_model_call 钩子：重试、降级、缓存]
#    ↓ [after_model 钩子：内容审核]
# 3. 工具执行
#    ↓ [wrap_tool_call 钩子：工具调用重试]
# 4. 回到模型思考（循环直到完成）
#    ↓ [after_agent 钩子：结果格式化、统计分析]
# 5. 输出结果
```

LangChain 的 Middleware 提供了 6 个钩子

| 钩子 | 执行频率 | 执行位置 | 主要用途 |
| :--- | :--- | :--- | :--- |
| **before_agent** | 一次 | Agent 开始前 | 初始化、权限检查、输入预处理 |
| **before_model** | 每次循环 | 模型调用前 | 消息预处理、动态上下文注入 |
| **wrap_model_call** | 每次循环 | 包裹模型调用 | 重试、降级、缓存、请求改写 |
| **after_model** | 每次循环 | 模型调用后 | 内容审核、响应过滤、日志 |
| **wrap_tool_call** | 每次工具调用 | 包裹工具执行 | 工具重试、结果缓存、参数改写 |
| **after_agent** | 一次 | Agent 结束后 | 格式化输出、统计、清理资源 |
